# TPUForge Multi-Chip Kaggle TPU v5e-8, Colab TPU v5e-1, & Cloud TPU v6e/v7 Benchmark

This notebook builds the native C++20 `tpuforge` runtime (`_tpuforge_ext` + `tpuforge_bench`) and benchmarks **Google Gemma 4** (`Apache 2.0`, ungated) plus cross-architecture OpenXLA kernels on Cloud TPU silicon.

### Flagship Gemma 4 Lineup & Hardware Mapping
- **Sparse MoE Flagship (`google/gemma-4-26B-A4B-it` + `z-lab/gemma-4-26B-A4B-it-DFlash`):** `128` experts, top-`8`, `4B` active per token. Fits in `32 GiB` `v6e-1` HBM at `INT8` (`25.08 GiB` total), scales `TP=8` on `v5e-8` in `BF16` or `INT8`, and streams experts via host `mmap` on `16 GiB` `v5e-1` (`--host-store mmap --stage-experts 16`).
- **Single-Chip Dense (`google/gemma-4-12B-it` + `z-lab/gemma4-12B-it-DFlash`):** Quantizes to `INT8` (`13.29 GiB` + `~0.68 GiB` DFlash) inside `16 GiB` `v5e-1` HBM, and runs both `BF16` and `INT8` on `32 GiB` `v6e-1` (`128` vs `256` systolic tile sweep).
- **Multi-Chip Dense (`google/gemma-4-31B-it` + `z-lab/gemma-4-31B-it-DFlash`):** Reuses tied `embed_tokens` as `lm_head` (`TP=8` `9.80 GiB/chip` `BF16` across `v5e-8`), paired with on-device `jnp.argmax` to eliminate `262,144`-vocab `134 MB/step` (`tile=128`) or `268 MB/step` (`tile=256`) PCIe logits transfers.
- **Optical Torus ICI Ring-Attention (`M-005`) & Secondary Cross-Arch Check:** Benchmarks `32K` / `128K` Ring-Attention plus `Qwen/Qwen3-4B` + `z-lab/Qwen3-4B-DFlash-b16` with zero-KV C++ `SuffixIndex` AST seeding (`--ast-seed-path`).

In [ ]:
# Cell 1: Select scratch mount, extract TPUForge bundle or clone repo, build C++20 runtime, probe TPU
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

best_scratch = max([p for p in ("/tmp", "/dev/shm", "/var/tmp") if os.path.isdir(p)], key=lambda p: shutil.disk_usage(p).free)
hf_cache = f"{best_scratch}/hf_cache"
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"] = hf_cache
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
print(f"Selected HF_HOME={os.environ['HF_HOME']} ({shutil.disk_usage(best_scratch).free / (1024**3):.1f} GiB free)")

RESULTS_DIR = (
    Path("/kaggle/working/tpuforge_results")
    if Path("/kaggle/working").exists()
    else Path("/tmp/tpuforge_results")
).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _run_checked(cmd, cwd=None):
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command exited {rc}: {' '.join(cmd)}")

def _reclaim_hf_cache(min_free_gib: int, label: str) -> float:
    hf_home = os.environ["HF_HOME"]
    free_gib = shutil.disk_usage(hf_home).free / (1024**3)
    print(f"\n=== [{label}] Free space on {hf_home}: {free_gib:.1f} GiB ===")
    if free_gib < min_free_gib:
        print(f"Reclaiming {hf_home} (below {min_free_gib} GiB threshold) before {label}...")
        shutil.rmtree(hf_home, ignore_errors=True)
        os.makedirs(hf_home, exist_ok=True)
        free_gib = shutil.disk_usage(hf_home).free / (1024**3)
        print(f"Free space after reclaim: {free_gib:.1f} GiB")
    return free_gib

missing_pkgs = []
for cli_name in ("cmake", "ninja"):
    if shutil.which(cli_name) is None:
        missing_pkgs.append(cli_name)
for mod_name in ("nanobind", "huggingface_hub", "tokenizers", "ml_dtypes"):
    if importlib.util.find_spec(mod_name) is None:
        missing_pkgs.append(mod_name)
if missing_pkgs:
    print(f"Installing missing build/runtime deps: {missing_pkgs}")
    _run_checked([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
else:
    print("All build/runtime dependencies already present; keeping preinstalled JAX/libtpu untouched.")

REPO_DIR = Path("/tmp/TPUForge")
build_dir = REPO_DIR / "build"
bundle_candidates = [
    Path("/tmp/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/input/tpuforge-src/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/working/tpuforge_kaggle_bundle.zip"),
    Path("./tpuforge_kaggle_bundle.zip"),
]
bundle_zip = next((p for p in bundle_candidates if p.is_file()), None)
if bundle_zip is not None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted TPUForge bundle from {bundle_zip} -> {REPO_DIR}")
elif Path("CMakeLists.txt").is_file() and Path("src/speculative/py_bindings.cpp").is_file():
    REPO_DIR = Path(".").resolve()
    build_dir = REPO_DIR / "build"
    print(f"Using current working checkout: {REPO_DIR}")
else:
    repo_url = os.environ.get("TPUFORGE_REPO_URL", "https://github.com/adamm285/TPUForge-dev.git")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)
need_rebuild = True
if need_rebuild:
    _run_checked([
        "cmake", "-S", str(REPO_DIR), "-B", str(build_dir),
        "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Release",
        f"-DPython_EXECUTABLE={sys.executable}",
    ])
    _run_checked(["cmake", "--build", str(build_dir)])

os.chdir(REPO_DIR)
for p_str in (str(REPO_DIR), str(build_dir)):
    if p_str not in sys.path:
        sys.path.insert(0, p_str)
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR), str(build_dir), os.environ.get("PYTHONPATH", "")]
).strip(os.pathsep)

# Probe TPU hardware inside an isolated subprocess so the parent kernel never locks /dev/vfio/*
probe_script = build_dir / "_hw_probe.py"
probe_json = RESULTS_DIR / "hw_probe.json"
probe_script.write_text(
    "import json, sys\n"
    "from pathlib import Path\n"
    f"sys.path[:0] = [{str(REPO_DIR)!r}, {str(build_dir)!r}]\n"
    "import jax\n"
    "import _tpuforge_ext as tpu\n"
    "assert hasattr(tpu.SystolicTreeVerifier, 'verify_greedy_tokens')\n"
    "assert hasattr(tpu.TreeBuilder, 'build_from_block_top_k')\n"
    "devs = jax.devices()\n"
    "dev_kind = str(getattr(devs[0], 'device_kind', devs[0]))\n"
    "def _infer_per_device_hbm_bytes(dev, kind: str) -> int:\n"
    "    try:\n"
    "        stats = dev.memory_stats()\n"
    "        if isinstance(stats, dict) and stats.get('bytes_limit'):\n"
    "            return int(stats['bytes_limit'])\n"
    "    except Exception:\n"
    "        pass\n"
    "    low = kind.lower()\n"
    "    if 'v6' in low or 'v7' in low or 'trillium' in low or '8x' in low:\n"
    "        return 32 * (1024**3)\n"
    "    return 16 * (1024**3)\n"
    "per_dev_hbm = _infer_per_device_hbm_bytes(devs[0], dev_kind)\n"
    "per_dev_hbm_gib = per_dev_hbm / (1024**3)\n"
    "total_hbm_gib = (len(devs) * per_dev_hbm) / (1024**3)\n"
    "auto_mxu = int(tpu.infer_mxu_dim(dev_kind))\n"
    "primary_model = 'google/gemma-4-26B-A4B-it' if (len(devs) >= 4 or total_hbm_gib >= 28.0) else 'google/gemma-4-12B-it'\n"
    "info = {'jax_version': jax.__version__, 'backend': jax.default_backend(), 'num_devices': len(devs), 'device_kind': dev_kind, 'per_dev_hbm_gib': round(per_dev_hbm_gib, 3), 'total_hbm_gib': round(total_hbm_gib, 3), 'auto_mxu': auto_mxu, 'primary_model': primary_model}\n"
    f"Path({str(probe_json)!r}).write_text(json.dumps(info, indent=2), encoding='utf-8')\n"
    "print(f'JAX {jax.__version__} | backend={jax.default_backend()} | devices={len(devs)}x {dev_kind} | HBM={per_dev_hbm_gib:.1f} GiB/dev ({total_hbm_gib:.1f} GiB total) | auto MXU={auto_mxu} | primary={primary_model}')\n",
    encoding="utf-8",
)
_run_checked([sys.executable, str(probe_script)], cwd=REPO_DIR)
HW_INFO = json.loads(probe_json.read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])
AUTO_MXU = int(HW_INFO["auto_mxu"])
PRIMARY_MODEL = str(HW_INFO["primary_model"])

In [ ]:
# Cell 2: Milestone M-005 Optical Torus Ring-Attention over ICI (4k parity + 32k + 128k on 8x TPU)
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
_run_checked([
    sys.executable, "-m", "benchmarks.ring_attention_jax",
    "--max-seq-len", "4096",
    "--valid-seq-len", "3072",
    "--ring-size", "auto",
    "--tp-size", "1",
    "--tile-size", "auto",
    "--check-parity",
    "--output-json", str(RESULTS_DIR / "m005_ring_4k_parity.json"),
], cwd=REPO_DIR)

_run_checked([
    sys.executable, "-m", "benchmarks.ring_attention_jax",
    "--max-seq-len", "32768",
    "--valid-seq-len", "28672",
    "--ring-size", "auto",
    "--tp-size", "1",
    "--tile-size", "auto",
    "--no-check-parity",
    "--output-json", str(RESULTS_DIR / "m005_ring_attention.json"),
], cwd=REPO_DIR)

if int(HW_INFO["num_devices"]) >= 8:
    _run_checked([
        sys.executable, "-m", "benchmarks.ring_attention_jax",
        "--max-seq-len", "131072",
        "--valid-seq-len", "114688",
        "--ring-size", "8",
        "--tp-size", "1",
        "--tile-size", "auto",
        "--no-check-parity",
        "--output-json", str(RESULTS_DIR / "m005_ring_128k.json"),
    ], cwd=REPO_DIR)
else:
    print(f"Skipping 128k 8-chip ring run: only {HW_INFO['num_devices']} device(s) visible.")

In [ ]:
# Cell 3: Google Gemma 4 12B Dense + DFlash Block-Tree + AST Seeding + 128 vs 256 Tile Sweep
import json
import shutil
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])
AUTO_MXU = int(HW_INFO["auto_mxu"])
tp_arg = "auto" if NUM_DEVICES > 1 else "1"

_reclaim_hf_cache(30, "Gemma 4 12B Dense + DFlash")

if TOTAL_HBM_GIB >= 28.0:
    bf16_path = RESULTS_DIR / "gemma4_12b_bf16_block_tree.json"
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-12B-it",
        "--draft-model", "z-lab/gemma4-12B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "none",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", tp_arg,
        "--output-json", str(bf16_path),
    ], cwd=REPO_DIR)

    int8_path = RESULTS_DIR / "gemma4_12b_int8_block_tree.json"
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-12B-it",
        "--draft-model", "z-lab/gemma4-12B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "int8",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", tp_arg,
        "--output-json", str(int8_path),
    ], cwd=REPO_DIR)

    tile128_path = RESULTS_DIR / "gemma4_12b_tile128.json"
    tile256_path = RESULTS_DIR / "gemma4_12b_tile256.json"
    if AUTO_MXU == 128:
        shutil.copyfile(bf16_path, tile128_path)
        alt_tile, alt_path = "256", tile256_path
    else:
        shutil.copyfile(bf16_path, tile256_path)
        alt_tile, alt_path = "128", tile128_path
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-12B-it",
        "--draft-model", "z-lab/gemma4-12B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "none",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", alt_tile,
        "--tp-size", tp_arg,
        "--output-json", str(alt_path),
    ], cwd=REPO_DIR)
else:
    int8_path = RESULTS_DIR / "gemma4_12b_int8_block_tree.json"
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-12B-it",
        "--draft-model", "z-lab/gemma4-12B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "int8",
        "--max-seq-len", "1024",
        "--max-new-tokens", "64",
        "--tile-size", "auto",
        "--tp-size", "1",
        "--output-json", str(int8_path),
    ], cwd=REPO_DIR)

    alt_tile = "256" if AUTO_MXU == 128 else "128"
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-12B-it",
        "--draft-model", "z-lab/gemma4-12B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "int8",
        "--max-seq-len", "1024",
        "--max-new-tokens", "64",
        "--tile-size", alt_tile,
        "--tp-size", "1",
        "--output-json", str(RESULTS_DIR / f"gemma4_12b_int8_tile{alt_tile}.json"),
    ], cwd=REPO_DIR)

In [ ]:
# Cell 4: Flagship Sparse MoE Google Gemma 4 26B-A4B (128 Experts, Top-8) + DFlash
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])

if NUM_DEVICES >= 8:
    _reclaim_hf_cache(60, "Gemma 4 26B-A4B MoE TP=8 BF16 + INT8")
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-26B-A4B-it",
        "--draft-model", "z-lab/gemma-4-26B-A4B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "none",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "8",
        "--output-json", str(RESULTS_DIR / "gemma4_26b_a4b_tp8_bf16.json"),
    ], cwd=REPO_DIR)
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-26B-A4B-it",
        "--draft-model", "z-lab/gemma-4-26B-A4B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "int8",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "8",
        "--output-json", str(RESULTS_DIR / "gemma4_26b_a4b_tp8_int8.json"),
    ], cwd=REPO_DIR)
elif TOTAL_HBM_GIB >= 28.0:
    _reclaim_hf_cache(55, "Gemma 4 26B-A4B MoE Single-Chip INT8 + DFlash")
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-26B-A4B-it",
        "--draft-model", "z-lab/gemma-4-26B-A4B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--weight-quant", "int8",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "1",
        "--output-json", str(RESULTS_DIR / "gemma4_26b_a4b_int8_hbm.json"),
    ], cwd=REPO_DIR)
else:
    free_gib = _reclaim_hf_cache(55, "Gemma 4 26B-A4B MoE mmap Streaming (16 GiB HBM)")
    if free_gib >= 55.0:
        _run_checked([
            sys.executable, "-m", "benchmarks.run_qwen_bench",
            "--model", "google/gemma-4-26B-A4B-it",
            "--host-store", "mmap",
            "--stage-experts", "16",
            "--weight-quant", "int8",
            "--ast-seed-path", str(REPO_DIR),
            "--max-seq-len", "1024",
            "--max-new-tokens", "64",
            "--tile-size", "auto",
            "--tp-size", "1",
            "--output-json", str(RESULTS_DIR / "gemma4_26b_a4b_int8_mmap.json"),
        ], cwd=REPO_DIR)
    else:
        status_note = {
            "model_id": "google/gemma-4-26B-A4B-it",
            "status": "skipped_insufficient_disk",
            "free_gib": round(free_gib, 2),
            "required_gib": 55.0,
        }
        (RESULTS_DIR / "gemma4_26b_a4b_int8_mmap_status.json").write_text(
            json.dumps(status_note, indent=2), encoding="utf-8"
        )
        print(f"Skipping 26B-A4B mmap download: {free_gib:.1f} GiB free < 55 GiB required.")

In [ ]:
# Cell 5: Google Gemma 4 31B Dense + DFlash via Megatron TP=8 (Pooled 8x HBM)
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
if NUM_DEVICES >= 8:
    _reclaim_hf_cache(70, "Gemma 4 31B Dense TP=8 + DFlash")
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "google/gemma-4-31B-it",
        "--draft-model", "z-lab/gemma-4-31B-it-DFlash",
        "--ast-seed-path", str(REPO_DIR),
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "8",
        "--output-json", str(RESULTS_DIR / "gemma4_31b_tp8_block_tree.json"),
    ], cwd=REPO_DIR)
else:
    print(f"Skipping Gemma 4 31B TP=8 benchmark: only {NUM_DEVICES} device(s) visible.")

In [ ]:
# Cell 6: Secondary Cross-Architecture Check (Qwen/Qwen3-4B + DFlash + AST Seeding)
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
tp_arg = "auto" if NUM_DEVICES > 1 else "1"

_reclaim_hf_cache(18, "Qwen3-4B Cross-Architecture Block-Tree Check")
_run_checked([
    sys.executable, "-m", "benchmarks.run_qwen_bench",
    "--model", "Qwen/Qwen3-4B",
    "--draft-model", "z-lab/Qwen3-4B-DFlash-b16",
    "--ast-seed-path", str(REPO_DIR),
    "--max-seq-len", "2048",
    "--max-new-tokens", "128",
    "--tile-size", "auto",
    "--tp-size", tp_arg,
    "--top-k", "8",
    "--output-json", str(RESULTS_DIR / "qwen3_4b_block_tree.json"),
], cwd=REPO_DIR)

In [ ]:
# Cell 7: Consolidated JSON Telemetry Dump
import json
from pathlib import Path

results_dir = (
    RESULTS_DIR
    if "RESULTS_DIR" in globals()
    else (
        Path("/kaggle/working/tpuforge_results")
        if Path("/kaggle/working").exists()
        else Path("/tmp/tpuforge_results")
    )
)
all_reports = {}
for json_file in sorted(results_dir.glob("*.json")):
    all_reports[json_file.stem] = json.loads(json_file.read_text(encoding="utf-8"))

print("=== BEGIN_TPUFORGE_JSON ===")
print(json.dumps(all_reports, indent=2))
print("=== END_TPUFORGE_JSON ===")